# 14e -- Expanded CPU Sweep On 14c TTA Features

Kernel slug: `yahiaakhalafallah/14e-tta-fusion-aqe-fic-sweep`.

This CPU-only notebook reuses the completed 14c Stage 2 TTA checkpoint and reruns Stages 3-5 for a two-block sweep. Block A runs the fine `w_tertiary` by `similarity_threshold` grid around the 14d C3 optimum. If A8 reproduces 14d C3 and the Block A best is not worse than C3, Block B sweeps AQE/FIC at the Block A best. It does not repeat detection, tracking, or feature extraction. Internal frame IDs remain the 0-based Stage 1/2 convention; MOT output is converted to 1-based in Stage 5.

In [ ]:
import os, sys, subprocess, shutil, json, time, tarfile
from datetime import datetime
from pathlib import Path

import numpy as np

print(f"Python : {sys.version.split()[0]}")
try:
    import torch
    print(f"PyTorch: {torch.__version__}")
    print(f"CUDA   : {torch.cuda.is_available()}")
except Exception as exc:
    print(f"Torch import deferred/failed: {exc}")
print("14e is CPU-only: enable_gpu=false in kernel-metadata.json")

## 1. Clone Repo And Install CPU Dependencies

In [ ]:
REPO_URL = "https://github.com/MRKDaGods/gp.git"
WORK_DIR = Path("/kaggle/working")
PROJECT = WORK_DIR / "gp"

if not PROJECT.exists():
    print(f"Cloning {REPO_URL} ...")
    subprocess.check_call(["git", "clone", "--depth", "1", "-b", "feature/pretrained-ensemble", REPO_URL, str(PROJECT)])
else:
    print("Repo already present; pulling latest ...")
    subprocess.check_call(["git", "-C", str(PROJECT), "pull", "--ff-only"])

os.chdir(str(PROJECT))
sys.path.insert(0, str(PROJECT))
print(f"Repo ready at {PROJECT}")

In [ ]:
def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("faiss-cpu", "motmetrics", "loguru", "omegaconf", "rich", "networkx>=3.1", "click")
pip("filterpy", "ftfy", "lapx", "timm")
pip("--no-deps", "ultralytics")
pip("--no-deps", "boxmot==11.0.3")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], cwd=str(PROJECT))
print("CPU dependencies installed")

In [ ]:
FAILED = []
for label, module in [
    ("faiss", "faiss"),
    ("motmetrics", "motmetrics"),
    ("omegaconf", "omegaconf"),
    ("networkx", "networkx"),
    ("cv2", "cv2"),
]:
    try:
        __import__(module)
        print(f"  OK {label}")
    except ImportError as exc:
        print(f"  MISSING {label}: {exc}")
        FAILED.append(label)
if FAILED:
    raise RuntimeError(f"Missing modules: {FAILED}")
print("Required downstream modules importable")

## 2. Mount CityFlowV2 Ground Truth

In [ ]:
import re as regex

for mount in ["/tmp", "/kaggle/working"]:
    total, used, free = shutil.disk_usage(mount)
    print(f"{mount:16s} {free / 1024**3:.1f} GB free / {total / 1024**3:.1f} GB total")

candidate_mounts = [
    Path("/kaggle/input/data-aicity-2023-track-2"),
    Path("/kaggle/input/datasets/thanhnguyenle/data-aicity-2023-track-2"),
]
CITYFLOW_INPUT = next((path for path in candidate_mounts if path.exists()), None)
if CITYFLOW_INPUT is None:
    raise FileNotFoundError("CityFlowV2 dataset not found; attach thanhnguyenle/data-aicity-2023-track-2")
print(f"CityFlowV2 input: {CITYFLOW_INPUT}")

TMP_DATA = Path("/tmp/datasets")
TMP_DATA.mkdir(parents=True, exist_ok=True)
DATA_RAW_PARENT = PROJECT / "data" / "raw"
if not DATA_RAW_PARENT.is_symlink():
    if DATA_RAW_PARENT.exists():
        shutil.rmtree(DATA_RAW_PARENT)
    DATA_RAW_PARENT.parent.mkdir(parents=True, exist_ok=True)
    DATA_RAW_PARENT.symlink_to(TMP_DATA)

DATA_RAW = TMP_DATA / "cityflowv2"
DATA_RAW.mkdir(parents=True, exist_ok=True)
for split_dir in sorted(CITYFLOW_INPUT.iterdir()):
    if not split_dir.is_dir() or split_dir.name not in ("train", "validation", "test"):
        continue
    for scene_dir in sorted(split_dir.iterdir()):
        if not scene_dir.is_dir():
            continue
        for cam_dir in sorted(scene_dir.iterdir()):
            if not cam_dir.is_dir():
                continue
            flat_name = f"{scene_dir.name}_{cam_dir.name}"
            flat_dir = DATA_RAW / flat_name
            if not flat_dir.exists():
                flat_dir.symlink_to(cam_dir)

cam_pattern = regex.compile(r"^S\d{2}_c\d{3}$")
cams = sorted(path.name for path in DATA_RAW.iterdir() if path.is_dir() and cam_pattern.match(path.name))
print(f"CityFlowV2 ready: {len(cams)} cameras")
for cam in cams:
    print(f"  {cam}")

## 3. Resolve 14c TTA Checkpoint And 10a Fallback

In [ ]:
INPUT_ROOT = Path("/kaggle/input")


def find_input_dir(slug: str, owner_slug: str, hints=()) -> Path:
    direct = INPUT_ROOT / slug
    if direct.exists():
        return direct

    owner, _, kernel = owner_slug.partition("/")
    nested = INPUT_ROOT / "notebooks" / owner / kernel
    if nested.exists():
        return nested

    lowered_slug = slug.lower()
    lowered_hints = tuple(str(hint).lower() for hint in hints)
    direct_children = list(INPUT_ROOT.iterdir()) if INPUT_ROOT.exists() else []
    for path in direct_children:
        if not path.is_dir():
            continue
        name = path.name.lower()
        if lowered_slug in name or all(hint in name for hint in lowered_hints):
            return path

    for path in INPUT_ROOT.rglob("checkpoint.tar.gz") if INPUT_ROOT.exists() else []:
        parent_text = str(path.parent).lower()
        if lowered_slug in parent_text or all(hint in parent_text for hint in lowered_hints):
            return path.parent

    return direct


def find_kernel_output_file(owner_slug: str, filename: str, hints=()) -> Path:
    slug = owner_slug.split("/", 1)[1]
    input_dir = find_input_dir(slug, owner_slug, hints=hints)
    direct = input_dir / filename
    if direct.exists():
        print(f"Mounted {owner_slug}: {input_dir}")
        return direct

    if INPUT_ROOT.exists():
        for candidate in INPUT_ROOT.rglob(filename):
            parent_text = str(candidate.parent).lower()
            if slug.lower() in parent_text or all(str(hint).lower() in parent_text for hint in hints):
                print(f"Mounted {owner_slug}: {candidate.parent}")
                return candidate

    print(f"{filename} not found at {direct}; trying Kaggle API fallback for {owner_slug}")
    dl_dir = Path("/tmp") / f"kaggle_{slug}_download"
    dl_dir.mkdir(parents=True, exist_ok=True)
    pattern = "^" + filename.replace(".", r"\.") + "$"
    result = subprocess.run(
        [
            "kaggle",
            "kernels",
            "output",
            owner_slug,
            "--file-pattern",
            pattern,
            "-p",
            str(dl_dir),
        ],
        capture_output=True,
        text=True,
    )
    print(result.stdout)
    print(result.stderr)
    downloaded = dl_dir / filename
    if downloaded.exists() and downloaded.stat().st_size > 0:
        print(f"Downloaded {filename} from {owner_slug}")
        return downloaded

    visible = [str(path) for path in INPUT_ROOT.rglob(filename)] if INPUT_ROOT.exists() else []
    raise FileNotFoundError(f"Could not resolve {filename} for {owner_slug}. Visible matches: {visible[:20]}")


def extract_checkpoint(checkpoint_path: Path, extract_dir: Path) -> tuple[str, Path]:
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)
    print(f"Extracting {checkpoint_path} ({checkpoint_path.stat().st_size / 1024**2:.1f} MB)")
    with tarfile.open(str(checkpoint_path), "r:gz") as tar:
        tar.extractall(str(extract_dir))

    metadata_path = extract_dir / "run_metadata.json"
    if metadata_path.exists():
        previous_meta = json.loads(metadata_path.read_text(encoding="utf-8"))
        run_name = previous_meta["run_name"]
        return run_name, extract_dir / run_name

    run_dirs = [path for path in extract_dir.iterdir() if path.is_dir() and (path / "stage2").exists()]
    if len(run_dirs) == 1:
        return run_dirs[0].name, run_dirs[0]
    raise FileNotFoundError(f"No run_metadata.json or unique stage2 run dir found in {extract_dir}")


SOURCE_14C = "yahiaakhalafallah/14c-tta-stage2"
SOURCE_10A = "yahiaakhalafallah/mtmc-10a-stages-0-2"

checkpoint_14c = find_kernel_output_file(SOURCE_14C, "checkpoint.tar.gz", hints=("14c", "tta", "stage2"))
SOURCE_14C_RUN_NAME, SOURCE_14C_RUN_DIR = extract_checkpoint(checkpoint_14c, Path("/tmp/14c_checkpoint"))
print(f"Loaded 14c run: {SOURCE_14C_RUN_NAME}")
for stage in ["stage1", "stage2"]:
    stage_dir = SOURCE_14C_RUN_DIR / stage
    print(f"  14c {stage}: exists={stage_dir.exists()} files={len([p for p in stage_dir.rglob('*') if p.is_file()]) if stage_dir.exists() else 0}")

SOURCE_10A_RUN_NAME = None
SOURCE_10A_RUN_DIR = None
if not (SOURCE_14C_RUN_DIR / "stage1").exists():
    checkpoint_10a = find_kernel_output_file(SOURCE_10A, "checkpoint.tar.gz", hints=("10a", "stages", "0", "2"))
    SOURCE_10A_RUN_NAME, SOURCE_10A_RUN_DIR = extract_checkpoint(checkpoint_10a, Path("/tmp/10a_checkpoint"))
    print(f"Loaded 10a fallback run: {SOURCE_10A_RUN_NAME}")

## 4. Reconstruct Stage 2 Features From 14c

In [ ]:
from src.core.data_models import TrackletFeatures
from src.core.io_utils import (
    load_embeddings,
    load_hsv_features,
    load_multi_query_embeddings,
    load_tracklets_by_camera,
)
from src.core.config import load_config, save_config
from src.core.logging_utils import setup_logging

DATA_OUT = Path("/tmp/pipeline_outputs")
DATA_OUT.mkdir(parents=True, exist_ok=True)
RUN_NAME = f"run_14e_tta_sweep_expanded_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
RUN_DIR = DATA_OUT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

stage1_source = SOURCE_14C_RUN_DIR / "stage1"
if not stage1_source.exists() and SOURCE_10A_RUN_DIR is not None:
    stage1_source = SOURCE_10A_RUN_DIR / "stage1"
stage2_source = SOURCE_14C_RUN_DIR / "stage2"

if not stage1_source.exists():
    raise FileNotFoundError(f"Tracklet metadata stage1 not found in 14c or 10a fallback: {stage1_source}")
if not stage2_source.exists():
    raise FileNotFoundError(f"14c Stage 2 features not found: {stage2_source}")

shutil.copytree(stage1_source, RUN_DIR / "stage1")
shutil.copytree(stage2_source, RUN_DIR / "stage2")

PRIMARY_EMBEDDINGS_PATH = RUN_DIR / "stage2" / "embeddings.npy"
TERTIARY_DINOV2_PATH = RUN_DIR / "stage2" / "embeddings_tertiary.npy"
for required in [PRIMARY_EMBEDDINGS_PATH, TERTIARY_DINOV2_PATH, RUN_DIR / "stage2" / "embedding_index.json", RUN_DIR / "stage2" / "hsv_features.npy"]:
    if not required.exists():
        raise FileNotFoundError(required)

embeddings, index_map = load_embeddings(RUN_DIR / "stage2")
hsv_features = load_hsv_features(RUN_DIR / "stage2")
multi_query_embeddings = load_multi_query_embeddings(RUN_DIR / "stage2", len(index_map))
tertiary_embeddings = np.load(TERTIARY_DINOV2_PATH)

if embeddings.shape[0] != len(index_map):
    raise ValueError(f"Primary embeddings/index_map mismatch: {embeddings.shape[0]} vs {len(index_map)}")
if hsv_features.shape[0] != len(index_map):
    raise ValueError(f"HSV/index_map mismatch: {hsv_features.shape[0]} vs {len(index_map)}")
if tertiary_embeddings.shape[0] != len(index_map):
    raise ValueError(f"Tertiary/index_map mismatch: {tertiary_embeddings.shape[0]} vs {len(index_map)}")

features = []
for row, item in enumerate(index_map):
    features.append(
        TrackletFeatures(
            track_id=int(item["track_id"]),
            camera_id=str(item["camera_id"]),
            class_id=int(item.get("class_id", 2)),
            embedding=embeddings[row].astype(np.float32),
            hsv_histogram=hsv_features[row].astype(np.float32),
            multi_query_embeddings=multi_query_embeddings[row],
        )
    )

tracklets_by_camera = load_tracklets_by_camera(RUN_DIR / "stage1")
setup_logging(level="INFO", log_file=RUN_DIR / "pipeline.log")

print(f"Run: {RUN_NAME}")
print(f"14c source run: {SOURCE_14C_RUN_NAME}")
print(f"Stage1 copied from: {stage1_source}")
print(f"Stage2 copied from: {stage2_source}")
print(f"Features: {len(features)} primary={embeddings.shape} tertiary={tertiary_embeddings.shape} hsv={hsv_features.shape}")
for camera_id in sorted(tracklets_by_camera):
    print(f"  {camera_id}: tracklets={len(tracklets_by_camera[camera_id])}")

## 5. Run Stages 3-5 For Expanded Sweep

In [ ]:
from src.stage3_indexing import run_stage3
from src.stage4_association import run_stage4
from src.stage5_evaluation import run_stage5

BLOCK_A_CONFIGS = [
    {"label": "A1", "block": "A", "w_tertiary": 0.45, "similarity_threshold": 0.48, "notes": "fine grid lower tertiary/lower threshold"},
    {"label": "A2", "block": "A", "w_tertiary": 0.45, "similarity_threshold": 0.50, "notes": "fine grid lower tertiary/production threshold"},
    {"label": "A3", "block": "A", "w_tertiary": 0.45, "similarity_threshold": 0.52, "notes": "fine grid lower tertiary/higher threshold"},
    {"label": "A4", "block": "A", "w_tertiary": 0.475, "similarity_threshold": 0.48, "notes": "fine grid mid-low tertiary/lower threshold"},
    {"label": "A5", "block": "A", "w_tertiary": 0.475, "similarity_threshold": 0.50, "notes": "fine grid mid-low tertiary/production threshold"},
    {"label": "A6", "block": "A", "w_tertiary": 0.475, "similarity_threshold": 0.52, "notes": "fine grid mid-low tertiary/higher threshold"},
    {"label": "A7", "block": "A", "w_tertiary": 0.50, "similarity_threshold": 0.48, "notes": "14d C3 tertiary/lower threshold"},
    {"label": "A8", "block": "A", "w_tertiary": 0.50, "similarity_threshold": 0.50, "notes": "replicate 14d C3"},
    {"label": "A9", "block": "A", "w_tertiary": 0.50, "similarity_threshold": 0.52, "notes": "14d C3 tertiary/higher threshold"},
    {"label": "A10", "block": "A", "w_tertiary": 0.525, "similarity_threshold": 0.48, "notes": "fine grid mid-high tertiary/lower threshold"},
    {"label": "A11", "block": "A", "w_tertiary": 0.525, "similarity_threshold": 0.50, "notes": "fine grid mid-high tertiary/production threshold"},
    {"label": "A12", "block": "A", "w_tertiary": 0.525, "similarity_threshold": 0.52, "notes": "fine grid mid-high tertiary/higher threshold"},
]

BLOCK_B_CANDIDATES = [
    {"label": "B1", "block": "B", "aqe_k": 2, "fic_regularisation": 0.50, "notes": "AQE lower k at Block A best"},
    {"label": "B2", "block": "B", "aqe_k": 4, "fic_regularisation": 0.50, "notes": "AQE higher k at Block A best"},
    {"label": "B3", "block": "B", "aqe_k": 3, "fic_regularisation": 0.30, "notes": "lower FIC regularisation at Block A best"},
    {"label": "B4", "block": "B", "aqe_k": 3, "fic_regularisation": 0.70, "notes": "higher FIC regularisation at Block A best"},
]

BASE_AQE_K = 3
BASE_FIC_REG = 0.50
C3_REPRO_TARGET = 0.77155
C3_REPRO_TOL = 0.001
SOLVER = "cc"
ALGORITHM = "conflict_free_cc"
LOUVAIN_RES = 0.70
APPEARANCE_WEIGHT = 0.70
HSV_WEIGHT = 0.0
ST_WEIGHT = round(1.0 - APPEARANCE_WEIGHT - HSV_WEIGHT, 4)
BRIDGE_PRUNE = 0.0
MAX_COMP_SIZE = 12
GALLERY_THRESH = 0.48
ORPHAN_MATCH_THRESH = 0.38
INTRA_MERGE = True
INTRA_MERGE_THRESH = 0.80
INTRA_MERGE_GAP = 30
MULTI_QUERY_WEIGHT = 0.00
CAMERA_BIAS = False
CAMERA_BIAS_ITERS = 2
ZONE_MODEL = False
ZONE_BONUS = 0.06
ZONE_PENALTY = 0.04
HIERARCHICAL = False
HIER_CENTROID_TH = 0.45
HIER_MERGE_TH = 0.45
HIER_ORPHAN_TH = 0.40
RERANKING = False
RERANKING_K1 = 20
RERANKING_K2 = 6
RERANKING_LAMBDA = 0.5
CAMERA_PAIR_NORM = False
AFLINK_ENABLED = False
AFLINK_MAX_TIME_GAP_FRAMES = 150
AFLINK_MAX_SPATIAL_GAP_PX = 150.0
AFLINK_MIN_DIRECTION_COS = 0.85
AFLINK_MIN_VELOCITY_RATIO = 0.5
AFLINK_VELOCITY_WINDOW = 5
MTMC_ONLY = False

GT_DIR = DATA_RAW
expected_cams = ["S01_c001", "S01_c002", "S01_c003", "S02_c006", "S02_c007", "S02_c008"]
if not any((GT_DIR / cam / "gt" / "gt.txt").exists() for cam in expected_cams):
    dataset_gt = Path("/kaggle/input/data-aicity-2023-track-2")
    extracted_gt = Path("/tmp/14c_checkpoint") / "gt_annotations"
    if dataset_gt.exists():
        GT_DIR = dataset_gt
    elif extracted_gt.exists():
        GT_DIR = extracted_gt
    else:
        raise FileNotFoundError("CityFlowV2 ground-truth annotations not found")


def load_metrics(report_path: Path) -> dict:
    if not report_path.exists():
        return {}
    payload = json.loads(report_path.read_text(encoding="utf-8"))
    details = payload.get("details", {}) or {}
    error_analysis = details.get("error_analysis", {}) or {}
    return {
        "mtmc_idf1": payload.get("mtmc_idf1", details.get("mtmc_idf1", payload.get("idf1"))),
        "trackeval_idf1": payload.get("idf1"),
        "mota": payload.get("mota"),
        "hota": payload.get("hota"),
        "id_switches": payload.get("id_switches"),
        "conflations": error_analysis.get("conflated_pred"),
        "fragmentations": error_analysis.get("fragmented_gt"),
        "num_pred_ids": payload.get("num_pred_ids", error_analysis.get("total_pred")),
    }


def build_overrides(config: dict, config_run_name: str) -> list[str]:
    w_tertiary = float(config["w_tertiary"])
    sim_thresh = float(config["similarity_threshold"])
    aqe_k = int(config.get("aqe_k", BASE_AQE_K))
    fic_reg = float(config.get("fic_regularisation", BASE_FIC_REG))
    return [
        f"project.run_name={config_run_name}",
        f"project.output_dir={DATA_OUT}",
        f"stage0.input_dir={DATA_RAW}",
        "stage0.cameras=[S01_c001,S01_c002,S01_c003,S02_c006,S02_c007,S02_c008]",
        f"stage4.association.query_expansion.k={aqe_k}",
        "stage4.association.query_expansion.alpha=5.0",
        "stage4.association.query_expansion.dba=false",
        f"stage4.association.graph.similarity_threshold={sim_thresh}",
        f"stage4.association.solver={SOLVER}",
        f"stage4.association.graph.algorithm={ALGORITHM}",
        f"stage4.association.graph.louvain_resolution={LOUVAIN_RES}",
        f"stage4.association.graph.bridge_prune_margin={BRIDGE_PRUNE}",
        f"stage4.association.graph.max_component_size={MAX_COMP_SIZE}",
        f"stage4.association.weights.vehicle.appearance={APPEARANCE_WEIGHT}",
        f"stage4.association.weights.vehicle.hsv={HSV_WEIGHT}",
        f"stage4.association.weights.vehicle.spatiotemporal={ST_WEIGHT}",
        "stage4.association.mutual_nn.top_k_per_query=20",
        "stage4.association.fic.enabled=true",
        f"stage4.association.fic.regularisation={fic_reg}",
        f"stage4.association.reranking.enabled={str(RERANKING).lower()}",
        f"stage4.association.reranking.k1={RERANKING_K1}",
        f"stage4.association.reranking.k2={RERANKING_K2}",
        f"stage4.association.reranking.lambda_value={RERANKING_LAMBDA}",
        f"stage4.association.camera_pair_norm.enabled={str(CAMERA_PAIR_NORM).lower()}",
        "stage4.association.fac.enabled=false",
        f"stage4.association.multi_query.enabled={str(MULTI_QUERY_WEIGHT > 0.0).lower()}",
        f"stage4.association.multi_query.weight={MULTI_QUERY_WEIGHT}",
        f"stage4.association.aflink.enabled={str(AFLINK_ENABLED).lower()}",
        f"stage4.association.aflink.max_time_gap_frames={AFLINK_MAX_TIME_GAP_FRAMES}",
        f"stage4.association.aflink.max_spatial_gap_px={AFLINK_MAX_SPATIAL_GAP_PX}",
        f"stage4.association.aflink.min_direction_cos={AFLINK_MIN_DIRECTION_COS}",
        f"stage4.association.aflink.min_velocity_ratio={AFLINK_MIN_VELOCITY_RATIO}",
        f"stage4.association.aflink.velocity_window={AFLINK_VELOCITY_WINDOW}",
        "stage4.association.secondary_embeddings.path=",
        "stage4.association.secondary_embeddings.weight=0.0",
        f"stage4.association.tertiary_embeddings.path={TERTIARY_DINOV2_PATH}",
        f"stage4.association.tertiary_embeddings.weight={w_tertiary}",
        f"stage4.association.fusion.w_tertiary={w_tertiary}",
        f"stage4.association.camera_bias.enabled={str(CAMERA_BIAS).lower()}",
        f"stage4.association.camera_bias.iterations={CAMERA_BIAS_ITERS}",
        f"stage4.association.zone_model.enabled={str(ZONE_MODEL).lower()}",
        "stage4.association.zone_model.zone_data_path=configs/datasets/cityflowv2_zones.json",
        f"stage4.association.zone_model.bonus={ZONE_BONUS}",
        f"stage4.association.zone_model.penalty={ZONE_PENALTY}",
        f"stage4.association.hierarchical.enabled={str(HIERARCHICAL).lower()}",
        f"stage4.association.hierarchical.centroid_threshold={HIER_CENTROID_TH}",
        f"stage4.association.hierarchical.merge_threshold={HIER_MERGE_TH}",
        f"stage4.association.hierarchical.orphan_threshold={HIER_ORPHAN_TH}",
        "stage4.association.hierarchical.max_merge_size=12",
        f"stage4.association.intra_camera_merge.enabled={str(INTRA_MERGE).lower()}",
        f"stage4.association.intra_camera_merge.threshold={INTRA_MERGE_THRESH}",
        f"stage4.association.intra_camera_merge.max_time_gap={INTRA_MERGE_GAP}",
        "stage4.association.gallery_expansion.enabled=true",
        f"stage4.association.gallery_expansion.threshold={GALLERY_THRESH}",
        f"stage4.association.gallery_expansion.orphan_match_threshold={ORPHAN_MATCH_THRESH}",
        "stage4.association.weights.length_weight_power=0.3",
        "stage4.association.temporal_overlap.enabled=true",
        "stage4.association.temporal_overlap.bonus=0.05",
        "stage4.association.temporal_overlap.max_mean_time=5.0",
        f"stage5.mtmc_only_submission={str(MTMC_ONLY).lower()}",
        "stage5.stationary_filter.enabled=true",
        "stage5.stationary_filter.min_displacement_px=150",
        "stage5.stationary_filter.max_mean_velocity_px=2.0",
        "stage5.min_submission_confidence=0.15",
        "stage5.cross_id_nms_iou=0.40",
        "stage5.min_trajectory_confidence=0.30",
        "stage5.min_trajectory_frames=40",
        "stage5.track_edge_trim.enabled=false",
        "stage5.track_smoothing.enabled=false",
        "stage5.gt_frame_clip=true",
        "stage5.gt_zone_filter=true",
        f"stage5.ground_truth_dir={GT_DIR}",
    ]


def run_config(config: dict) -> dict:
    label = config["label"]
    config_run_name = f"{RUN_NAME}_{label}"
    config_dir = RUN_DIR / label
    config_dir.mkdir(parents=True, exist_ok=True)
    aqe_k = int(config.get("aqe_k", BASE_AQE_K))
    fic_reg = float(config.get("fic_regularisation", BASE_FIC_REG))
    print("\n" + "=" * 80)
    print(
        f"Running {label}: block={config['block']} w_tertiary={config['w_tertiary']:.3f}, "
        f"similarity_threshold={config['similarity_threshold']:.2f}, aqe_k={aqe_k}, fic_reg={fic_reg:.2f}"
    )
    print("=" * 80)

    cfg = load_config("configs/default.yaml", dataset_config="configs/datasets/cityflowv2.yaml", overrides=build_overrides(config, config_run_name))
    save_config(cfg, config_dir / "config.yaml")

    start = time.time()
    faiss_index, metadata_store = run_stage3(cfg, features, tracklets_by_camera, output_dir=config_dir / "stage3")
    stage3_min = (time.time() - start) / 60.0
    print(f"{label} Stage 3 complete in {stage3_min:.2f} min")

    start = time.time()
    trajectories = run_stage4(cfg, faiss_index, metadata_store, features, tracklets_by_camera, output_dir=config_dir / "stage4")
    stage4_min = (time.time() - start) / 60.0
    print(f"{label} Stage 4 complete in {stage4_min:.2f} min: {len(trajectories)} global trajectories")

    start = time.time()
    evaluation_result = run_stage5(cfg, trajectories, output_dir=config_dir / "stage5")
    stage5_min = (time.time() - start) / 60.0
    print(f"{label} Stage 5 complete in {stage5_min:.2f} min")

    report_path = config_dir / "stage5" / "evaluation_report.json"
    metrics = load_metrics(report_path)
    idf1_value = metrics.get("mtmc_idf1")
    if idf1_value is None:
        idf1_value = metrics.get("trackeval_idf1")
    if idf1_value is None:
        raise RuntimeError(f"IDF1 not found in {report_path}")

    row = {
        "label": label,
        "block": config["block"],
        "w_tertiary": float(config["w_tertiary"]),
        "w_primary": round(1.0 - float(config["w_tertiary"]), 6),
        "similarity_threshold": float(config["similarity_threshold"]),
        "aqe_k": aqe_k,
        "fic_regularisation": fic_reg,
        "notes": config["notes"],
        "idf1": idf1_value,
        "mtmc_idf1": metrics.get("mtmc_idf1"),
        "trackeval_idf1": metrics.get("trackeval_idf1"),
        "mota": metrics.get("mota"),
        "hota": metrics.get("hota"),
        "id_switches": metrics.get("id_switches"),
        "conflations": metrics.get("conflations"),
        "fragmentations": metrics.get("fragmentations"),
        "num_pred_ids": metrics.get("num_pred_ids"),
        "stage_timings_min": {
            "stage3": round(stage3_min, 2),
            "stage4": round(stage4_min, 2),
            "stage5": round(stage5_min, 2),
        },
        "paths": {
            "config_dir": str(config_dir),
            "evaluation_report": str(report_path),
        },
    }
    print(f"{label} MTMC IDF1: {idf1_value:.5f}")
    return row


results = []
block_a_results = []
block_b_results = []
halt_reason = None
block_b_skipped_reason = None
wall_start = time.time()

for config in BLOCK_A_CONFIGS:
    row = run_config(config)
    results.append(row)
    block_a_results.append(row)
    (RUN_DIR / "14e_partial_results.json").write_text(json.dumps(results, indent=2), encoding="utf-8")

    if row["label"] == "A8" and abs(float(row["idf1"]) - C3_REPRO_TARGET) > C3_REPRO_TOL:
        halt_reason = f"A8 failed to reproduce 14d C3: got {row['idf1']:.5f}, expected {C3_REPRO_TARGET:.5f} +/- {C3_REPRO_TOL:.5f}"
        print(halt_reason)
        break

best_block_a = max(block_a_results, key=lambda row: row["idf1"] if row["idf1"] is not None else -1.0)

if halt_reason is None:
    if float(best_block_a["idf1"]) < C3_REPRO_TARGET:
        block_b_skipped_reason = f"Block A best {best_block_a['idf1']:.5f} is worse than 14d C3 control {C3_REPRO_TARGET:.5f}"
        print(block_b_skipped_reason)
    else:
        block_b_configs = []
        for candidate in BLOCK_B_CANDIDATES:
            block_b_configs.append({
                **candidate,
                "w_tertiary": float(best_block_a["w_tertiary"]),
                "similarity_threshold": float(best_block_a["similarity_threshold"]),
                "notes": f"{candidate['notes']} ({best_block_a['label']} w_t={best_block_a['w_tertiary']:.3f}, thr={best_block_a['similarity_threshold']:.2f})",
            })

        for config in block_b_configs:
            row = run_config(config)
            results.append(row)
            block_b_results.append(row)
            (RUN_DIR / "14e_partial_results.json").write_text(json.dumps(results, indent=2), encoding="utf-8")

overall_best = max(results, key=lambda row: row["idf1"] if row["idf1"] is not None else -1.0)
best = overall_best
print("\n" + "#" * 80)
print(
    f"BEST 14e CONFIG: {overall_best['label']} block={overall_best['block']} "
    f"w_tertiary={overall_best['w_tertiary']:.3f} sim_thresh={overall_best['similarity_threshold']:.2f} "
    f"aqe_k={overall_best['aqe_k']} fic_reg={overall_best['fic_regularisation']:.2f} "
    f"MTMC_IDF1={overall_best['idf1']:.5f}"
)
if halt_reason:
    print(f"HALT: {halt_reason}")
if block_b_skipped_reason:
    print(f"BLOCK B SKIPPED: {block_b_skipped_reason}")
print("#" * 80)

## 6. Save 14e Summary

In [ ]:
checkpoint_path = Path("/kaggle/working/checkpoint.tar.gz")
metadata_path = Path("/kaggle/working/run_metadata.json")
summary_path = Path("/kaggle/working/14e_summary.json")

summary = {
    "run_name": RUN_NAME,
    "experiment": "14e-tta-fusion-aqe-fic-sweep",
    "kernel": "yahiaakhalafallah/14e-tta-fusion-aqe-fic-sweep",
    "source_14c_kernel": SOURCE_14C,
    "source_14c_run_name": SOURCE_14C_RUN_NAME,
    "source_10a_kernel": SOURCE_10A,
    "source_10a_run_name": SOURCE_10A_RUN_NAME,
    "frame_id_convention": "0-based internal Stage 1/2 frame IDs; MOT output is converted to 1-based in Stage 5",
    "control_reproduction_target": C3_REPRO_TARGET,
    "control_reproduction_tolerance": C3_REPRO_TOL,
    "blocks": {
        "block_a": {
            "description": "Fine w_tertiary by similarity_threshold grid on 14c TTA features",
            "grid": BLOCK_A_CONFIGS,
            "results": block_a_results,
            "best": best_block_a,
            "grid_size": len(BLOCK_A_CONFIGS),
        },
        "block_b": {
            "description": "AQE/FIC sweep at the Block A best point",
            "grid_template": BLOCK_B_CANDIDATES,
            "results": block_b_results,
            "best": max(block_b_results, key=lambda row: row["idf1"] if row["idf1"] is not None else -1.0) if block_b_results else None,
            "grid_size": len(BLOCK_B_CANDIDATES),
            "skipped_reason": block_b_skipped_reason,
        },
    },
    "results": results,
    "overall_best": overall_best,
    "best": overall_best,
    "halt_reason": halt_reason,
    "feature_sources": {
        "stage1_tracklets": str(stage1_source),
        "stage2_features": str(stage2_source),
        "primary_embeddings": str(PRIMARY_EMBEDDINGS_PATH),
        "tertiary_embeddings": str(TERTIARY_DINOV2_PATH),
        "num_features": len(features),
        "primary_shape": list(embeddings.shape),
        "tertiary_shape": list(tertiary_embeddings.shape),
        "hsv_shape": list(hsv_features.shape),
    },
    "fixed_config": {
        "base_aqe_k": BASE_AQE_K,
        "base_fic_regularisation": BASE_FIC_REG,
        "pca_components": 384,
        "algorithm": ALGORITHM,
        "gallery_expansion": True,
        "gallery_threshold": GALLERY_THRESH,
        "orphan_match_threshold": ORPHAN_MATCH_THRESH,
        "temporal_overlap_bonus": 0.05,
        "mtmc_only_submission": MTMC_ONLY,
    },
    "stop_criteria": {
        "win_threshold": 0.7720,
        "marginal_min": 0.7715,
        "neutral_min": 0.7705,
        "dead_below": 0.7705,
    },
    "walltime_min": round((time.time() - wall_start) / 60.0, 2),
    "paths": {
        "run_dir": str(RUN_DIR),
        "summary": str(summary_path),
    },
}

metadata_path.write_text(json.dumps({"run_name": RUN_NAME}, indent=2), encoding="utf-8")
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(f"Building checkpoint for {RUN_NAME}")
with tarfile.open(str(checkpoint_path), "w:gz") as tar:
    tar.add(str(metadata_path), arcname="run_metadata.json")
    tar.add(str(summary_path), arcname="14e_summary.json")
    for file_path in RUN_DIR.rglob("*"):
        if file_path.is_file() and "stage2" not in file_path.parts and "stage1" not in file_path.parts:
            tar.add(str(file_path), arcname=f"{RUN_NAME}/{file_path.relative_to(RUN_DIR)}")

size_mb = checkpoint_path.stat().st_size / 1024**2
print(f"Saved summary: {summary_path}")
print(f"Checkpoint: {checkpoint_path} ({size_mb:.1f} MB)")
print(json.dumps(summary, indent=2))